**Technical Workshop** - Python Notebook **2/2**

[HRFest 2026](https://hrfest.org)

[Miguel A. Solis](https://www.miguelsolis.info)

07 de noviembre de 2026

In [ ]:
!sudo apt-get install -y python-opengl > /dev/null 2>&1
!pip install git+https://github.com/tensorflow/docs > /dev/null 2>&1

In [ ]:
import gymnasium as gym
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers
import random
from collections import deque
import matplotlib.pyplot as plt
from matplotlib import animation
import IPython

class ReplayBuffer:
    def __init__(self, capacity=2000):
        self.buffer = deque(maxlen=capacity)

    def guardar(self, estado, accion, recompensa, sig_estado, fin):
        self.buffer.append((estado, accion, recompensa, sig_estado, fin))

    def muestra(self, largo):
        batch = random.sample(self.buffer, largo)
        estados, acciones, recompensas, sig_estados, finales = zip(*batch)
        return np.array(estados), np.array(acciones), np.array(recompensas, dtype=np.float32), np.array(sig_estados), np.array(finales, dtype=np.float32)

In [ ]:
def crear_modelo(num_estados, num_acciones):
    modelo = tf.keras.Sequential([
        layers.Input(shape=(num_estados,)),
        layers.Dense(64, activation='relu'),
        layers.Dense(64, activation='relu'),
        layers.Dense(num_acciones, activation='linear')
    ])
    return modelo

In [ ]:
class DQN:
    def __init__(self, num_estados, num_acciones):
        self.num_acciones = num_acciones
        self.modelo = crear_modelo(num_estados, num_acciones)
        self.modelo_objetivo = crear_modelo(num_estados, num_acciones)
        self.actualizar_red()

        self.optimizador = tf.keras.optimizers.Adam(learning_rate=0.001)
        self.memoria = ReplayBuffer()

        self.gamma = 0.95
        self.epsilon = 1.0
        self.epsilon_min = 0.01
        self.epsilon_decaimiento = 0.995

    def actualizar_red(self):
        self.modelo_objetivo.set_weights(self.modelo.get_weights())

    def actuar(self, estado):
        if np.random.rand() <= self.epsilon:
            return random.randrange(self.num_acciones)
        valores_q = self.modelo.predict(estado[np.newaxis], verbose=0)
        return np.argmax(valores_q[0])

    def entrenar(self, largo):
        if len(self.memoria.buffer) < largo:
            return

        estados, acciones, recompensas, sig_estados, finales = self.memoria.muestra(largo)

        with tf.GradientTape() as tape:
            valores_q = self.modelo(estados)
            acciones_codificadas = tf.one_hot(acciones, self.num_acciones)
            q_actual = tf.reduce_sum(valores_q * acciones_codificadas, axis=1)

            sig_valores_q = self.modelo_objetivo(sig_estados)
            max_sig_q = tf.reduce_max(sig_valores_q, axis=1)
            q_objetivo = recompensas + (self.gamma * max_sig_q * (1 - finales))

            perdida = tf.keras.losses.MSE(q_objetivo, q_actual)

        gradientes = tape.gradient(perdida, self.modelo.trainable_variables)
        self.optimizador.apply_gradients(zip(gradientes, self.modelo.trainable_variables))

        if self.epsilon > self.epsilon_min:
            self.epsilon *= self.epsilon_decaimiento

In [ ]:
env = gym.make("CartPole-v1")
num_observaciones = int(env.observation_space.shape[0])
num_acciones = int(env.action_space.n)

agente = DQN(num_observaciones, num_acciones)

episodios = 100
largo = 32
recompensas_historicas = []

for e in range(episodios):
    estado, _ = env.reset()
    recompensa_total = 0

    for t in range(500):
        accion = agente.actuar(estado)
        sig_estado, recompensa, terminado, truncado, _ = env.step(accion)
        fin = terminado or truncado

        agente.memoria.guardar(estado, accion, recompensa, sig_estado, fin)
        estado = sig_estado
        recompensa_total += recompensa

        if fin:
            agente.actualizar_red()
            print(f"Episodio: {e}/{episodios}, Puntaje: {recompensa_total}, Epsilon: {agente.epsilon:.2f}")
            break

        agente.entrenar(largo)

    recompensas_historicas.append(recompensa_total)

In [ ]:
plt.plot(recompensas_historicas)
plt.title("Aprendizaje del Agente")
plt.xlabel("Episodio")
plt.ylabel("Recompensa Total")
plt.show()

In [ ]:
def mostrar_animacion_entorno(cuadros, title="Video"):
    if not cuadros:
        print("Error: No hay fotogramas para mostrar.")
        return

    fig = plt.figure(figsize=(6, 4))
    patch = plt.imshow(cuadros[0])
    plt.title(title)
    plt.axis('off')

    def animar(i):
        patch.set_data(cuadros[i])

    anim = animation.FuncAnimation(fig, animar, frames=len(cuadros), interval=50)

    video_html = IPython.display.HTML(anim.to_jshtml())
    IPython.display.display(video_html)

    plt.close()

In [ ]:
env_final = gym.make("CartPole-v1", render_mode="rgb_array")
estado, _ = env_final.reset()
cuadros_finales = []

for t in range(100):
    cuadros_finales.append(env_final.render())

    valores_q = agente.modelo.predict(estado[np.newaxis], verbose=0)
    accion = np.argmax(valores_q[0])
    estado, recompensa, terminado, truncado, _ = env_final.step(accion)

    if terminado or truncado:
        break

env_final.close()

mostrar_animacion_entorno(cuadros_finales, "Péndulo Entrenado")

In [ ]:
env_aleatorio = gym.make("CartPole-v1", render_mode="rgb_array")
state, _ = env_aleatorio.reset()
cuadros_aleatorios = []

for t in range(100):
    cuadros_aleatorios.append(env_aleatorio.render())

    accion = env_aleatorio.action_space.sample()
    estado, recompensa, terminado, truncado, _ = env_aleatorio.step(accion)

    if terminado or truncado:
        break

env_aleatorio.close()

mostrar_animacion_entorno(cuadros_aleatorios, "Péndulo Aleatorio")